# 04 - Audit Dataset Quality (Blocking)

Notebook bloquant: accepte ou refuse le dataset, sans correction automatique.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = Path('../data/classifier_dataset')
MASTER = BASE / 'processed' / 'dataset_master.csv'
AUDIT = BASE / 'audit_reports'
AUDIT.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    'query_id','text_raw','text_clean','label_primary','expected_decision','risk_tags','icu_topic',
    'source_dataset','source_type','is_synthetic','family_id','difficulty','requires_context',
    'contains_pii','contains_patient_data','split','review_status','annotator_1','annotator_2',
    'final_label','notes'
]

VALID_REVIEW = {'reviewed', 'validated'}
VALID_SPLITS = {'train', 'val', 'test', 'frozen_safety_test', ''}
MIN_FAMILY_PER_LABEL = 200
MAX_SIMILARITY_RATIO = 0.05

## Load and schema checks

In [2]:
if not MASTER.exists():
    raise FileNotFoundError(f'Missing dataset_master.csv: {MASTER}')

df = pd.read_csv(MASTER)
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

assert df['family_id'].notna().all(), 'family_id contains null values'
assert df['expected_decision'].notna().all(), 'expected_decision contains null values'
assert df['source_dataset'].notna().all(), 'source_dataset contains null values'

split_invalid = set(df['split'].fillna('').unique()) - VALID_SPLITS
if split_invalid:
    raise ValueError(f'Invalid split values: {split_invalid}')

print('Rows:', len(df))
print('Columns OK')

Rows: 175829
Columns OK


## Blocking quality rules

In [3]:
# Rule 1: reviewed/validated only in non-empty splits
curated = df[df['split'].astype(str).str.len() > 0].copy()
if curated.empty:
    raise ValueError('Dataset refused: no curated rows (split assigned) found. Complete annotation/review/splitting first.')

bad_review = ~curated['review_status'].isin(VALID_REVIEW)
if bad_review.any():
    bad = curated.loc[bad_review, ['query_id','split','review_status']].head(20)
    bad.to_csv(AUDIT / 'invalid_review_status.csv', index=False)
    raise ValueError('Dataset refused: split rows must be reviewed/validated')

# Rule 2: no synthetic rows in val/test/frozen sets
for s in ['val', 'test', 'frozen_safety_test']:
    part = curated[curated['split'] == s]
    if not part.empty and part['is_synthetic'].astype(str).str.lower().isin(['true','1']).mean() > 0:
        raise ValueError(f'Dataset refused: {s} contains synthetic rows')

# Rule 3: minimum families per label (on curated rows only)
fam_counts = curated.groupby('label_primary')['family_id'].nunique().sort_values()
fam_counts.to_csv(AUDIT / 'label_distribution.csv', header=['n_families'])
if int(fam_counts.min()) < MIN_FAMILY_PER_LABEL:
    raise ValueError(
        f'Dataset refused: min families per label is {int(fam_counts.min())}, expected >= {MIN_FAMILY_PER_LABEL}'
    )

# Rule 4: split leakage by family_id (on curated rows)
train_f = set(curated[curated['split'] == 'train']['family_id'])
val_f = set(curated[curated['split'] == 'val']['family_id'])
test_f = set(curated[curated['split'] == 'test']['family_id'])
if train_f & val_f or train_f & test_f or val_f & test_f:
    raise ValueError('Dataset refused: family_id leakage across train/val/test')

# Rule 5: template concentration by family (on curated rows)
family_share = curated['family_id'].value_counts(normalize=True)
if family_share.iloc[0] > 0.20:
    raise ValueError(f'Dataset refused: top family share is {family_share.iloc[0]:.2%} > 20%')

print('Blocking rules passed')

Blocking rules passed


## Duplicate and semantic leakage reports

In [4]:
def norm(t: str) -> str:
    t = unicodedata.normalize('NFKD', str(t))
    t = ''.join(ch for ch in t if not unicodedata.combining(ch))
    t = t.lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

work_df = curated.copy()
work_df['_norm'] = work_df['text_clean'].fillna(work_df['text_raw']).map(norm)
dups = work_df[work_df.duplicated(subset=['_norm', 'label_primary'], keep=False)].sort_values(['label_primary','_norm'])
dups.to_csv(AUDIT / 'duplicates_report.csv', index=False)

# Cross-split semantic leakage (TF-IDF cosine)
train_df = work_df[work_df['split'] == 'train']
test_df = work_df[work_df['split'] == 'test']

if not train_df.empty and not test_df.empty:
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    X_train = vec.fit_transform(train_df['_norm'])
    X_test = vec.transform(test_df['_norm'])
    sim = cosine_similarity(X_test, X_train)
    max_sim = sim.max(axis=1)
    leakage_ratio = float((max_sim >= 0.90).mean())

    sem = pd.DataFrame({
        'query_id': test_df['query_id'].values,
        'max_similarity_to_train': max_sim,
    }).sort_values('max_similarity_to_train', ascending=False)
    sem.to_csv(AUDIT / 'semantic_leakage_report.csv', index=False)

    if leakage_ratio > MAX_SIMILARITY_RATIO:
        raise ValueError(
            f'Dataset refused: semantic leakage ratio={leakage_ratio:.2%} > {MAX_SIMILARITY_RATIO:.2%}'
        )
else:
    pd.DataFrame(columns=['query_id','max_similarity_to_train']).to_csv(AUDIT / 'semantic_leakage_report.csv', index=False)

source_dist = work_df['source_dataset'].value_counts().rename_axis('source_dataset').reset_index(name='count')
source_dist.to_csv(AUDIT / 'source_distribution.csv', index=False)

print('Saved audit reports in', AUDIT)
print('- duplicates_report.csv')
print('- semantic_leakage_report.csv')
print('- source_distribution.csv')
print('- label_distribution.csv')

Saved audit reports in ..\data\classifier_dataset\audit_reports
- duplicates_report.csv
- semantic_leakage_report.csv
- source_distribution.csv
- label_distribution.csv


## Final status

In [5]:
summary = [
    '# Dataset Audit Summary',
    '',
    f'- Rows: {len(df)}',
    f'- Labels: {sorted(df["label_primary"].dropna().unique().tolist())}',
    '- Status: ACCEPTED if no exception was raised above.',
]
(AUDIT / 'audit_summary.md').write_text('\n'.join(summary), encoding='utf-8')
print('Audit summary written to', AUDIT / 'audit_summary.md')

Audit summary written to ..\data\classifier_dataset\audit_reports\audit_summary.md
